In [12]:
import os

from deepfix_sdk import DeepFixClient

In [ ]:
os.environ["DEEPFIX_API_KEY"] = "sk-empty"

In [14]:
client = DeepFixClient(api_url="https://deepfix.delcaux.com", timeout=120)

# Tabular data

## Classification

In [15]:
from deepfix_sdk.data.datasets import TabularDataset
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

In [16]:
X, y = load_breast_cancer(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
dataset_name = "breast_cancer_classification"

label = "target"
train = X_train.copy()
train[label] = y_train
cat_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
if len(cat_features) > 0:
    cat_features = None

test = X_test.copy()
test[label] = y_test

train_data = TabularDataset(
    dataset=train, dataset_name=dataset_name, label=label, cat_features=cat_features
)
val_data = TabularDataset(
    dataset=test, dataset_name=dataset_name, label=label, cat_features=cat_features
)

# Fit model
model_name = "HistGradientBoostingClassifier"
clf = HistGradientBoostingClassifier(max_depth=3)
clf = clf.fit(train_data.X, train_data.y)

2025-11-14 22:21:37 WARNI [deepfix_sdk.data.datasets] No categorical features provided, will automatically detect them. (Not Recommended)
2025-11-14 22:21:37 WARNI [deepfix_sdk.data.datasets] No categorical features provided, will automatically detect them. (Not Recommended)


In [17]:
train_data.get_data().head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
546,10.32,16.35,65.31,324.9,0.09434,0.04994,0.01012,0.005495,0.1885,0.06201,...,21.77,71.12,384.9,0.1285,0.08842,0.04384,0.02381,0.2681,0.07399,1
432,20.18,19.54,133.80,1250.0,0.11330,0.14890,0.21330,0.125900,0.1724,0.06053,...,25.07,146.00,1479.0,0.1665,0.29420,0.53080,0.21730,0.3032,0.08075,0
174,10.66,15.15,67.49,349.6,0.08792,0.04302,0.00000,0.000000,0.1928,0.05975,...,19.20,73.20,408.3,0.1076,0.06791,0.00000,0.00000,0.2710,0.06164,1
221,13.56,13.90,88.59,561.3,0.10510,0.11920,0.07860,0.044510,0.1962,0.06303,...,17.13,101.10,686.6,0.1376,0.26980,0.25770,0.09090,0.3065,0.08177,1
289,11.37,18.89,72.17,396.0,0.08713,0.05008,0.02399,0.021730,0.2013,0.05955,...,26.14,79.29,459.3,0.1118,0.09708,0.07529,0.06203,0.3267,0.06994,1


In [ ]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    model_name=model_name,
    model=clf,
    language="english",
)

In [20]:
# Visualize results
result.to_text(verbose=False)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The cross-artifact analysis reveals a critical gap between theoretical model performance and deployment readiness.   │
│ While the Deepchecks analysis shows excellent model performance (AUC ~1.0) with some feature optimization            │
│ opportunities (high redundancy, unused features), the ModelCheckpoint analysis exposes that the model cannot be      │
│ deployed due to missing trained model files and essential metadata. The highest priority is generating the actual    │
│ model artifact. Once available, feature optimization should be addressed to improve model stability and leverage     │
│ unused high-variance features. The excellent performance metrics are promising but meaningless without executable    │
│ model files.                                                                                                         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  4                                                            
 Severity Distribution           MEDIUM: 2  HIGH: 1  LOW: 1                                   

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical deployment blocker: Missing     │ Generate and include the actual trained  │
│     │ trained model file                       │ model file (pickle, joblib, or ONNX      │
│     │ Evidence: ModelCheckpoint analyzer       │ format) with the configuration artifacts │
│     │ confirms no actual model weights or      │ The model configuration alone is         │
│     │ serialized model object exists, only     │ insufficient for deployment or           │
│     │ configuration parameters. Deepchecks     │ inference. The excellent performance     │
│     │ analysis shows excellent theoretical     │ metrics from Deepchecks are meaningless  │
│     │ performance (AUC ~1.0) but this cannot   │ without the executable model.            │
│     │ be utilized without the model file.      │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (2)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Incomplete model metadata hinders        │ Include feature_names_in_, classes_, and │
│     │ deployment                               │ n_features_in_ attributes with the       │
│     │ Evidence: Missing feature names, class   │ trained model, and review feature        │
│     │ labels, and feature count metadata in    │ selection strategy                       │
│     │ ModelCheckpoint, while Deepchecks shows  │ Proper metadata is essential for         │
│     │ 16 high-variance features are unused and │ deployment. The feature issues           │
│     │ 22 feature pairs have high correlation   │ (redundancy, unused features) from       │
│     │ (>0.9)                                   │ Deepchecks analysis should inform which  │
│     │                                          │ features to include in the final         │
│     │                                          │ deployed model.                          │
│ 2   │ Feature optimization opportunity despite │ Perform feature selection (PCA or        │
│     │ excellent performance                    │ representative feature selection) and    │
│     │ Evidence: Deepchecks shows high feature  │ investigate incorporating unused         │
│     │ redundancy (22 correlated pairs >0.9)    │ high-variance features                   │
│     │ and 16 unused high-variance features,    │ Reducing multicollinearity can improve   │
│     │ while model maintains AUC ~1.0 with F1   │ model stability and interpretability.    │
│     │ gain of 91.49%                           │ The unused features may contain valuable │
│     │                                          │ signal for robustness.                   │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                   LOW Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Excellent model performance with         │ Monitor performance on new data and      │
│     │ potential overfitting risk               │ conduct feature importance analysis to   │
│     │ Evidence: Deepchecks reports             │ validate robustness                      │
│     │ near-perfect performance (AUC ~1.0) but  │ While current performance is             │
│     │ with heavy reliance on 3 features        │ exceptional, reliance on few features    │
│     │ showing very high predictive power (PPS  │ may make the model vulnerable to         │
│     │ > 0.7)                                   │ distribution shifts in production.       │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

# Computer vision

## Image classification

In [11]:
from deepfix_sdk.data.datasets import ImageClassificationDataset
from deepfix_sdk.zoo.datasets.foodwaste import load_train_and_val_datasets

In [12]:
dataset_name = "cafetaria-foodwaste-lstroetmann"
# Load image datasets
train_data, val_data = load_train_and_val_datasets(
    image_size=448,
    batch_size=8,
    num_workers=4,
    pin_memory=False,
)
train_data = ImageClassificationDataset(dataset_name=dataset_name, dataset=train_data)
val_data = ImageClassificationDataset(dataset_name=dataset_name, dataset=val_data)

Getting label mapping: 100%|██████████| 375/375 [00:03<00:00, 107.85it/s]


In [ ]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    language="english",
)

Computing dataset base statistics: 100%|██████████| 160/160 [00:11<00:00, 14.06it/s]


In [15]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The cross-artifact analysis reveals catastrophic data quality issues that invalidate the current machine learning    │
│ setup. The test set suffers from severe label distribution drift (Cramer's V=0.92) and contains 75% new labels not   │
│ seen in training, indicating fundamental problems with data partitioning. Additionally, significant differences in   │
│ image properties suggest inconsistent acquisition conditions. These issues collectively mean that any model          │
│ evaluation would be unreliable. Immediate remediation of the data splitting methodology and image standardization is │
│ required before proceeding with model development.                                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  3                                                            
 Severity Distribution           HIGH: 2  MEDIUM: 1                                           

                                  HIGH Severity Issues (2)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical data partitioning failure       │ Immediately halt model development and   │
│     │ causing unrepresentative test set        │ recreate the train-test split using      │
│     │ Evidence: Combined evidence from         │ proper stratified sampling techniques    │
│     │ Deepchecks: Label drift check failed     │ The test set is fundamentally invalid    │
│     │ with Cramer's V score of 0.92 (far       │ for evaluation due to severe label       │
│     │ exceeding 0.15 threshold) and 75% of     │ distribution mismatch and leakage,       │
│     │ test set labels were not present in      │ making any model performance metrics     │
│     │ training                                 │ meaningless                              │
│ 2   │ Systematic differences in image          │ Standardize image collection protocols   │
│     │ acquisition conditions between datasets  │ and apply normalization techniques to    │
│     │ Evidence: Multiple image property drift  │ align visual characteristics             │
│     │ failures: Brightness (KS=0.42), RMS      │ Large differences in brightness,         │
│     │ Contrast (KS=0.5), Red Intensity         │ contrast, and color properties will      │
│     │ (KS=0.83), Green Intensity (KS=0.82),    │ cause models to learn dataset-specific   │
│     │ Blue Intensity (KS=0.96)                 │ artifacts rather than generalizable      │
│     │                                          │ features                                 │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (1)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Incomplete data quality assessment       │ Implement comprehensive data validation  │
│     │ framework                                │ pipeline including outlier detection,    │
│     │ Evidence: DatasetArtifactsAnalyzer       │ label consistency checks, and metadata   │
│     │ failed due to technical issues, and      │ validation                               │
│     │ Deepchecks data integrity section was    │ Current assessment gaps prevent          │
│     │ incomplete                               │ identification of additional data        │
│     │                                          │ quality issues that could impact model   │
│     │                                          │ reliability and performance              │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

## Object detection

In [16]:
from deepfix_sdk.data.datasets import ObjectDetectionDataset

In [17]:
dataset_name = "general_dataset"
train_data = ObjectDetectionDataset.from_coco(
    dataset_name=dataset_name,
    images_directory_path=r"D:\workspace\general_dataset\coco\train",
    annotations_path=r"D:\workspace\general_dataset\coco\annotations\annotations_train.json",
)
val_data = ObjectDetectionDataset.from_coco(
    dataset_name=dataset_name,
    images_directory_path=r"D:\workspace\general_dataset\coco\val",
    annotations_path=r"D:\workspace\general_dataset\coco\annotations\annotations_val.json",
)

In [ ]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    language="english",
)

Computing base box statistics: 100%|██████████| 668/668 [00:00<?, ?it/s]


In [20]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The consolidated analysis indicates that the primary risk of overfitting stems from data quality issues identified   │
│ by the Deepchecks analyzer, specifically an unstable feature-label relationship and significant brightness drift     │
│ between train and test sets. These issues suggest the model may overfit to spurious correlations and specific        │
│ lighting conditions. The failure of the DatasetArtifacts analyzer and the incomplete data integrity assessment in    │
│ the Deepchecks results mean that the overall data quality picture is incomplete, presenting a secondary,             │
│ medium-severity risk. Recommendations focus on addressing the identified instabilities and completing the missing    │
│ data integrity checks to ensure the model learns robust, generalizable patterns.                                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  2                                                            
 Severity Distribution           HIGH: 1  MEDIUM: 1                                           

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical data quality and stability      │ Prioritize mitigating the identified     │
│     │ issues identified as primary overfitting │ feature-label instability and brightness │
│     │ risks                                    │ drift. Investigate the technical error   │
│     │ Evidence: Deepchecks analysis revealed a │ to enable a full DatasetArtifacts        │
│     │ high-severity unstable feature-label     │ analysis for a comprehensive view.       │
│     │ relationship (PPS difference: 0.21) and  │ The model is at high risk of overfitting │
│     │ significant image brightness drift (KS   │ due to learning non-generalizable        │
│     │ score: 0.29). The DatasetArtifacts       │ correlations and being sensitive to      │
│     │ analysis was unavailable due to a system │ lighting variations. A complete dataset  │
│     │ error, preventing a complete dataset     │ analysis is needed to rule out other     │
│     │ assessment.                              │ underlying data issues.                  │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (1)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Incomplete data integrity validation     │ Run a full suite of data integrity       │
│     │ obscures potential data quality issues   │ checks, including outlier detection and  │
│     │ Evidence: The Deepchecks analysis noted  │ label validation, to identify any hidden │
│     │ an empty or missing data integrity       │ data quality problems.                   │
│     │ section, leaving outlier detection and   │ Unassessed data integrity issues can     │
│     │ label consistency unassessed. Combined   │ silently contribute to overfitting. A    │
│     │ with the failed DatasetArtifacts         │ complete assessment is crucial for       │
│     │ analysis, there is a gap in the overall  │ building a robust model.                 │
│     │ data quality evaluation.                 │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

## Semantic segmentation

In [21]:
from deepfix_sdk.data.datasets import SemanticSegmentationDataset
from deepfix_sdk.zoo.datasets import load_segmentation_dataset

In [22]:
dataset_name = "coco_segmentation"
train_data, val_data = load_segmentation_dataset(
    batch_size=8,
    shuffle=False,
    pin_memory=False,
)
train_data = SemanticSegmentationDataset(
    dataset_name=dataset_name, dataset=train_data.dataset
)
val_data = SemanticSegmentationDataset(
    dataset_name=dataset_name, dataset=val_data.dataset
)

In [ ]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    language="english",
)

Computing dataset base statistics: 100%|██████████| 49/49 [00:03<00:00, 14.83it/s]


In [25]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The analysis reveals critical data quality issues primarily identified through Deepchecks validation. The most       │
│ severe problems involve significant distribution mismatches between training and test datasets, including class      │
│ imbalance (0.17 categorical drift) and color property differences (red: 0.2, green: 0.24 drift scores). These        │
│ distribution inconsistencies threaten model reliability and generalization. Additionally, gaps in the data           │
│ validation framework (evidenced by incomplete integrity checks and analyzer failures) suggest a need for stronger    │
│ quality assurance processes. Immediate attention should focus on rebalancing datasets and implementing comprehensive │
│ validation to ensure model performance reflects true capabilities rather than dataset artifacts.                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  2                                                            
 Severity Distribution           HIGH: 1  MEDIUM: 1                                           

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical data distribution               │ Implement comprehensive data             │
│     │ inconsistencies between training and     │ distribution analysis and rebalance      │
│     │ validation sets                          │ training/validation splits to ensure     │
│     │ Evidence: Deepchecks analysis shows      │ consistent class and feature             │
│     │ categorical drift score of 0.17 for      │ distributions                            │
│     │ 'Samples Per Class' (exceeding 0.15      │ Distribution mismatches between datasets │
│     │ threshold) and color property drifts     │ lead to unreliable model evaluation and  │
│     │ (Mean Red: 0.2, Mean Green: 0.24         │ poor generalization to real-world data   │
│     │ exceeding 0.2 threshold), indicating     │                                          │
│     │ significant distribution mismatches      │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (1)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Insufficient data quality validation     │ Establish robust data validation         │
│     │ framework                                │ pipeline with comprehensive integrity    │
│     │ Evidence: Deepchecks analysis indicates  │ checks, outlier detection, and automated │
│     │ incomplete data integrity validation     │ quality monitoring                       │
│     │ section, and DatasetArtifactsAnalyzer    │ Missing or incomplete validation         │
│     │ failed due to technical issues,          │ increases risk of undetected data        │
│     │ suggesting gaps in the data validation   │ quality issues that can compromise model │
│     │ pipeline                                 │ performance and reliability              │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

# NLP

## Sentence classification

In [26]:
from deepfix_sdk.data.datasets import NLPDataset
from deepfix_sdk.zoo.datasets import load_tweet_emotion_classification

In [27]:
train_data, test_data = load_tweet_emotion_classification(
    as_train_test=True, include_embeddings=True
)
dataset_name = "tweet_emotion_classification"
train_data = NLPDataset(dataset_name=dataset_name, dataset=train_data)
test_data = NLPDataset(dataset_name=dataset_name, dataset=test_data)

In [ ]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    language="english",
)

deepchecks - WARNING - Could not find model's classes, using the observed classes. In order to make sure the classes used by the model are inferred correctly, please use the model_classes argument


In [30]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The cross-artifact analysis was partially successful. The DatasetArtifactsAnalyzer encountered a technical error and │
│ could not provide results. However, the DeepchecksArtifactsAnalyzer identified critical data quality issues. The     │
│ most severe finding is a high-confidence label distribution drift between train and test sets, which poses a         │
│ significant risk to model validity. Additional concerns include a high ratio of outliers in text toxicity and the    │
│ presence of unknown tokens, both rated as medium severity. A low-severity text embeddings drift was also noted.      │
│ Recommendations focus on data preprocessing, tokenizer updates, and careful performance monitoring to ensure model   │
│ robustness. The failure of one analyzer highlights a potential need to review the artifact analysis pipeline for     │
│ stability.                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  4                                                            
 Severity Distribution           MEDIUM: 2  HIGH: 1  LOW: 1                                   

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Significant label distribution drift     │ Re-examine the train-test split          │
│     │ between train and test sets              │ methodology to ensure proper             │
│     │ Evidence: Label drift check failed with  │ stratification                           │
│     │ Cramer's V score of 0.22, exceeding the  │ Label distribution mismatch can lead to  │
│     │ 0.15 threshold, indicating substantial   │ unreliable performance metrics and model │
│     │ distribution shift in emotion labels     │ overfitting to train-specific patterns   │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (2)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ High outlier ratio in text properties,   │ Investigate and clean outliers in the    │
│     │ particularly Toxicity                    │ Toxicity property, or implement robust   │
│     │ Evidence: Text property outliers check   │ preprocessing                            │
│     │ failed with Toxicity property showing    │ High outlier ratios can distort feature  │
│     │ 16.43% outlier ratio, significantly      │ relationships and model training,        │
│     │ above the 5% threshold                   │ potentially leading to poor              │
│     │                                          │ generalization                           │
│ 2   │ Presence of unknown tokens indicating    │ Update tokenizer vocabulary or           │
│     │ tokenizer coverage gaps                  │ preprocess text to handle unknown tokens │
│     │ Evidence: Unknown tokens check failed    │ appropriately                            │
│     │ with ratios of 0.79% and 0.68%,          │ Unknown tokens can degrade model         │
│     │ indicating unsupported tokens in the     │ performance and introduce noise in text  │
│     │ dataset                                  │ representations                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                   LOW Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Moderate text embeddings drift           │ Monitor model performance closely and    │
│     │ suggesting domain shift                  │ consider domain adaptation techniques if │
│     │ Evidence: Text embeddings drift showed   │ performance degrades                     │
│     │ AUC of 0.6, indicating some domain shift │ Domain shift can affect model            │
│     │ between train and test distributions     │ generalization, though the current level │
│     │                                          │ may be acceptable for deployment         │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''